# Computational Tools for Climate Science - Project 2026
---
**Group 2 - Chaac**

Authors: *Abiyo Gladys, Alex Blackmer, Caroline Roper, Han Nguyen, Negli Gallardo, Sheila Aguilar Palpa*

Date: *July 13-24th*

---

This project explores the relationship between precipitation and crop production in Southeast Asia. Specifically, we tie the Standardized Precipitation Index (SPI) derived from Climate Hazards Group InfraRed Precipitation with Station data (CHIRPS) with cereal production yield anomalies sourced from the Food and Agriculture Organization (FAO). This study considers two domains in Southeast Asia; the mainland (Monsoon Climate Region - MCR) and maritime (Equatorial Climate Region - ECR), focusing on the countries of Vietnam and Indonesia respectively.

In [131]:
# Install packages not included in base environment
!pip install cartopy
!pip install gdown
!pip install xclim

In [132]:
# Imports
import os
# Scientific data packages
import numpy as np
import xarray as xr
import pandas as pd
import datetime
# Plotting libraries
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.io.shapereader as shpreader
# Climate index package
import xclim
from scipy.stats import pearsonr
import pandas as pd
import statsmodels.api as sm
from statsmodels.formula.api import ols

---
# Data Management
## Mount Google Drive
First step is to mount the Google Drive fonder containing group project data

In [133]:
# Mount Google Drive project data path
from google.colab import drive
drive.mount('/content/drive')
path_gd =  "/content/drive/MyDrive/Data/"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Downloading data
This is a helper function for downloading data

---

### Download CHIRPS data

---
## Load Datasets
This section loads the country masks, precipitation, and cereal production data used for analysis

### Precipitation

In [134]:
path_gd = '/content/drive/MyDrive/Group Project Folder/Data/'

In [135]:
precip_ds = xr.open_dataset(path_gd+ "chirps_sel.nc")
precip_ds

<xarray.Dataset> Size: 2GB
Dimensions:    (time: 546, latitude: 900, longitude: 1200)
Coordinates:
  * time       (time) datetime64[ns] 4kB 1981-01-01 1981-02-01 ... 2026-06-01
  * latitude   (latitude) float32 4kB -19.98 -19.92 -19.88 ... 24.88 24.92 24.97
  * longitude  (longitude) float32 5kB 90.02 90.08 90.12 ... 149.9 149.9 150.0
Data variables:
    precip     (time, latitude, longitude) float32 2GB ...
Attributes: (12/14)
    Conventions:       CF-1.6
    title:             CHIRPS Version 3.0
    history:           created by Climate Hazards Center
    version:           Version 3.0
    data_created:      2026-07-14
    creator_name:      Pete Peterson
    ...                ...
    documentation:     http://pubs.usgs.gov/ds/832/
    reference:         Funk, C.C., Peterson, P.J., Landsfeld, M.F., Pedreros,...
    acknowledgements:  The Climate Hazards Center InfraRed Precipitation with...
    ftp_url:           ftp://chg-ftpout.geog.ucsb.edu/pub/chg/products/CHIRPS...
    website:           http://chg.geog.ucsb.edu/data/chirps/index.html
    faq:               http://chg-wiki.geog.ucsb.edu/wiki/CHIRPS_FAQ

### Country Mask

In [136]:
country_mask = xr.open_dataset(path_gd + 'CountryMask/NetCDF/CountryMergedNC.nc')
country_mask = country_mask.rename({"lat": "latitude", "lon": "longitude"})

country_mask = country_mask["country_mask"].interp(
    latitude=precip_ds.latitude,
    longitude=precip_ds.longitude,
    method="nearest"
)

### Cereal Production

In [137]:
cereal_df = pd.read_csv(path_gd + "data_cereal_land.csv", header=4)

---
## Subset Precipitation Data By Country

In [138]:
# Mask each contry area and drop the NaNs
precip_ind = precip_ds.precip.where(country_mask == 1, drop=True)
precip_viet = precip_ds.precip.where(country_mask == 2, drop=True)

---
# Analysis

## SPI Calculation
Standardized precipitation index is calculated individually for Vietnam and Indonesia

In [139]:
precip_viet_1d = precip_viet.mean(dim=["latitude", "longitude"])
spi_1m_viet = xclim.indicators.atmos.standardized_precipitation_index(pr=precip_viet_1d, freq='MS', window=1)
spi_6m_viet = xclim.indicators.atmos.standardized_precipitation_index(pr=precip_viet_1d, freq='MS', window=6)
spi_12m_viet = xclim.indicators.atmos.standardized_precipitation_index(pr=precip_viet_1d, freq='MS', window=12)

/usr/local/lib/python3.12/dist-packages/xclim/core/cfchecks.py:77: UserWarning: Variable does not have a `cell_methods` attribute.
  _check_cell_methods(getattr(vardata, "cell_methods", None), data["cell_methods"])
/usr/local/lib/python3.12/dist-packages/xclim/core/cfchecks.py:79: UserWarning: Variable has a non-conforming standard_name: Got `convective precipitation rate`, expected `['precipitation_flux']`
  check_valid(vardata, "standard_name", data["standard_name"])
/usr/local/lib/python3.12/dist-packages/xclim/core/cfchecks.py:77: UserWarning: Variable does not have a `cell_methods` attribute.
  _check_cell_methods(getattr(vardata, "cell_methods", None), data["cell_methods"])
/usr/local/lib/python3.12/dist-packages/xclim/core/cfchecks.py:79: UserWarning: Variable has a non-conforming standard_name: Got `convective precipitation rate`, expected `['precipitation_flux']`
  check_valid(vardata, "standard_name", data["standard_name"])
/usr/local/lib/python3.12/dist-packages/xclim/core/c

In [140]:
precip_ind_1d = precip_ind.mean(dim=["latitude", "longitude"])
spi_1m_ind = xclim.indicators.atmos.standardized_precipitation_index(pr=precip_ind_1d, freq='MS', window=1)
spi_6m_ind = xclim.indicators.atmos.standardized_precipitation_index(pr=precip_ind_1d, freq='MS', window=6)
spi_12m_ind = xclim.indicators.atmos.standardized_precipitation_index(pr=precip_ind_1d, freq='MS', window=12)

/usr/local/lib/python3.12/dist-packages/xclim/core/cfchecks.py:77: UserWarning: Variable does not have a `cell_methods` attribute.
  _check_cell_methods(getattr(vardata, "cell_methods", None), data["cell_methods"])
/usr/local/lib/python3.12/dist-packages/xclim/core/cfchecks.py:79: UserWarning: Variable has a non-conforming standard_name: Got `convective precipitation rate`, expected `['precipitation_flux']`
  check_valid(vardata, "standard_name", data["standard_name"])
/usr/local/lib/python3.12/dist-packages/xclim/core/cfchecks.py:77: UserWarning: Variable does not have a `cell_methods` attribute.
  _check_cell_methods(getattr(vardata, "cell_methods", None), data["cell_methods"])
/usr/local/lib/python3.12/dist-packages/xclim/core/cfchecks.py:79: UserWarning: Variable has a non-conforming standard_name: Got `convective precipitation rate`, expected `['precipitation_flux']`
  check_valid(vardata, "standard_name", data["standard_name"])
/usr/local/lib/python3.12/dist-packages/xclim/core/c

In [141]:
def monthly_severity_per_year(x):
  '''takes x, an xarray.DataArray with variable 'spi' monthly and outputs a pandas dataframe of the number of months at each severity level per year'''
  yearly = x.groupby('time.year').map(lambda x: x.groupby_bins(x, bins=[-np.inf, -2, -1.5, -1, 0, np.inf]).count()).fillna(0)
  yearly = yearly.to_dataframe().pivot_table(index = 'year', columns = 'spi_bins')
  yearly.columns = yearly.columns.droplevel(0)
  new_column_names = ['Extreme', 'Severe', 'Moderate', 'Mild', 'No Drought']
  yearly.columns = new_column_names
  return yearly

In [142]:
def categorize_spi(spi_data_array):
  """
  Categorizes SPI values into drought severity levels.

  Takes an xarray DataArray containing 12m SPI values with a 'time' coordinate.

  Returns a pandas DataFrame with the annual 'spi' and its 'category'
  """
  spi_12m_annual = spi_data_array.resample(time='1YE').last()
  spi_12m_annual_pd = spi_12m_annual.to_pandas()
  spi_12m_annual_pd.index = spi_12m_annual_pd.index.year
  spi_12m_annual_pd.index.name = 'year'
  spi_values = spi_12m_annual_pd.rename('SPI Value')

  # Define bins and labels for drought severity
  bins = [-np.inf, -2.0, -1.5, -1.0, 0, np.inf]
  labels = ['Extreme Drought', 'Severe Drought', 'Moderate Drought', 'Mild Drought', 'No Drought']

  # Categorize SPI values
  spi_category = pd.cut(spi_values, bins=bins, labels=labels, right=False)

  # Create a DataFrame from the results
  result_df = pd.DataFrame({
      'spi': spi_values,
      'category': spi_category
  })

  return result_df

In [143]:
per_year_months_at_severity_level_viet = monthly_severity_per_year(spi_1m_viet)

In [144]:
per_year_months_at_severity_level_indonesia = monthly_severity_per_year(spi_1m_ind)

In [145]:
spi_yearly_viet = categorize_spi(spi_12m_viet)
spi_yearly_indonesia = categorize_spi(spi_12m_ind)

In [146]:
annual_spi_viet = per_year_months_at_severity_level_viet.join(spi_yearly_viet)
#annual_spi_viet.to_csv(path_gd + 'annual_spi_vietnam.csv')

In [147]:
annual_spi_indonesia = per_year_months_at_severity_level_indonesia.join(spi_yearly_indonesia)
#annual_spi_indonesia.to_csv(path_gd + 'annual_spi_indonesia.csv')

## Cereal Anomalies

In [148]:
#Importing data
from google.colab import drive
drive.mount('/content/drive')
path =  "/content/drive/MyDrive/Group Project Folder/Data/"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [149]:
#Creating a dataset
cereal_land = pd.read_csv(path + "data_cereal_land.csv", skiprows=4, header=0)

In [150]:
#cereal_land

In [151]:
#Selectiong countries and columns of interest
countries = ['Indonesia', 'Viet Nam']
columns_of_interest = ['Country Name', 'Country Code'] + [str(year) for year in range(1981, 2024)]

In [152]:
cereal_land_countries = cereal_land[cereal_land['Country Name'].isin(countries)]

In [153]:
existing_columns = [col for col in columns_of_interest if col in cereal_land_countries.columns]
cereal_land_filtered = cereal_land_countries[existing_columns]

In [154]:
cereal_land_long_VI = cereal_land_filtered.melt(
    id_vars=['Country Name', 'Country Code'],
    var_name='Year',
    value_name='Production'
)

In [155]:
#Calculating the mean production of cereal
mean_production_country = cereal_land_long_VI.groupby('Country Name')['Production'].mean().reset_index()
mean_production_country = mean_production_country.rename(columns={'Production': 'Production_Mean'})

In [156]:
#Merging the mean to the dataset
cereal_land_long_VI = cereal_land_long_VI.merge(mean_production_country, on='Country Name')

In [157]:
#Calculating anomaly
cereal_land_long_VI['Anomaly'] = cereal_land_long_VI['Production'] - cereal_land_long_VI['Production_Mean']

Normalizing cereal production anomaly.

In [158]:
#Calculating standar deviation for each country (original time series)
std_by_country = cereal_land_long_VI.groupby('Country Name')['Production'].std().reset_index()
std_by_country = std_by_country.rename(columns={'Production': 'Production_Std'})

In [159]:
#Merge new variable
cereal_land_long_VI = cereal_land_long_VI.merge(std_by_country, on='Country Name')

In [160]:
#Normalizing anomaly (Z-score)
cereal_land_long_VI['Anomaly_Normalized'] = cereal_land_long_VI['Anomaly'] / cereal_land_long_VI['Production_Std']

In [161]:
#Save dataset as CSV file
path_drive = "/content/drive/MyDrive/Group Project Folder/Data/"
cereal_land_long_VI.to_csv(path_drive + 'normalized_anomaly_cereal_data.csv', index=False)

## SPI & Cereal Correlation
TODO: Individually correlate each SPI series (1m, 6m, 12m) with cereal anomalies to assess which has strongest relationship. Do for each domain.

## Drought Severity Analysis
TODO: Subset each SPI series by drought severity category to then assess relationship between drought severity and crop production. Can create scatter plots and time series. Do for each domain.

In [162]:
# crops_ind = cereal_land_long_VI.loc[cereal_land_long_VI['Country Name'] == 'Indonesia', :].set_index('Year') # this is our data that is NOT de-trended

In [163]:
detrended = pd.read_csv(path_gd + 'detrended_normalized_anomaly_cerial_data.csv')

In [164]:
def add_anomaly(df, colname):
  '''
  takes a pandas dataframe with a Country Name column and finds the anomaly and a normalized anomoly of the column given by "colname"
  '''
  mean_production_country = df.groupby('Country Name')[colname].mean().reset_index()
  mean_production_country = mean_production_country.rename(columns={colname: colname + '_Mean'})
  df = df.merge(mean_production_country, on='Country Name')
  df['Anomaly'] = df[colname] - df[colname + '_Mean']
  #Calculating standar deviation for each country (original time series)
  std_by_country = df.groupby('Country Name')[colname].std().reset_index()
  std_by_country = std_by_country.rename(columns={colname: colname + '_Std'})
  #Merge new variable
  df = df.merge(std_by_country, on='Country Name')
  #Normalizing anomaly (Z-score)
  df['Anomaly_Normalized'] = df['Anomaly'] / df[colname + '_Std']
  return df

In [165]:
detrended.head()

,Country Name,Country Code,Year,Production,detrended,detrended_Mean,Anomaly,detrended_Std,Anomaly_Normalized
0,Indonesia,IDN,1981,37283478.0,-1.433609e+06,4.851541e-09,-1.433609e+06,2.158616e+06,-0.664133
1,Viet Nam,VNM,1981,12844800.0,1.973415e+06,1.433804e-08,1.973415e+06,1.203843e+06,1.639262
2,Indonesia,IDN,1982,36818521.0,-3.127823e+06,4.851541e-09,-3.127823e+06,2.158616e+06,-1.448995
3,Viet Nam,VNM,1982,14828300.0,2.774720e+06,1.433804e-08,2.774720e+06,1.203843e+06,2.304885
4,Indonesia,IDN,1983,40389883.0,-7.857188e+05,4.851541e-09,-7.857188e+05,2.158616e+06,-0.363992


In [166]:
#detrended = add_anomaly(detrended, 'detrended')

In [182]:
def prep_corr(df, spi_df, country):
  '''takes a pandas dataframe with a "Country Name" column and a country name found within the "Country Name" column and provides a joined truncated dataframe of the relevant columns'''
  new_df = df.loc[df['Country Name'] == country].set_index('Year')
  new_df.index.name = 'year'
  new_df.index = new_df.index.astype('int')
  new_df = spi_df.join(new_df)
  new_df = new_df.loc[:2023, :]
  return new_df

In [186]:
crops_and_spi_ind = prep_corr(detrended, annual_spi_indonesia, 'Indonesia')

In [185]:
crops_and_spi_viet = prep_corr(detrended, annual_spi_viet, 'Viet Nam')

In [187]:
crops_and_spi_ind.groupby('category')['Anomaly_Normalized'].agg(['mean', 'median'])

/tmp/ipykernel_1409/738625448.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  crops_and_spi_ind.groupby('category')['Anomaly_Normalized'].agg(['mean', 'median'])


,mean,median
category,,
Extreme Drought,-0.109931,-0.109931
Severe Drought,-0.855763,-1.448995
Moderate Drought,-0.239427,-0.290132
Mild Drought,0.083517,-0.230239
No Drought,0.117414,0.256427


In [173]:
#independence assumption may be violated here

In [174]:
# Define the model
model = ols('Anomaly ~ C(category)', data=crops_and_spi_ind).fit()

# Perform ANOVA
anova_table = sm.stats.anova_lm(model, typ=1)
print(anova_table)

               df        sum_sq       mean_sq         F    PR(>F)
C(category)   4.0  1.316592e+13  3.291479e+12  0.685205  0.606621
Residual     38.0  1.825383e+14  4.803640e+12       NaN       NaN


In [175]:
crops_and_spi_ind[['spi', 'Anomaly_Normalized']].corr()

,spi,Anomaly_Normalized
spi,1.000000,0.279321
Anomaly_Normalized,0.279321,1.000000


In [177]:
correlation_coefficient, p_value = pearsonr(crops_and_spi_ind.spi.values, crops_and_spi_ind.Anomaly_Normalized.values)

In [178]:
pearsonr(spi, crops)

PearsonRResult(statistic=np.float64(0.27932142894026996), pvalue=np.float64(0.06968470803047949))

In [189]:
correlation_coefficient, p_value = pearsonr(crops_and_spi_viet.spi.values, crops_and_spi_viet.Anomaly_Normalized.values)
print (correlation_coefficient, p_value)

0.09779664378711148 0.5326953878258527
